# 11 — Vectors and Geometric Intuition

**Master syllabus coverage:** V2 2.2

## Why this matters

Tokens, hidden states, gradients, and parameter updates are vectors. Norm, angle, and projection give a language for their size, alignment, and decomposition.

## Learning objectives

- Implement dot products and norms from scalar operations.
- Distinguish Euclidean distance from cosine similarity.
- Project a vector and verify the orthogonal residual.
- Compute pairwise similarity for a batch of embeddings.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Vectors are coordinates and directions

A vector $x\in\mathbb{R}^d$ is an ordered list relative to a basis. In an embedding
space, coordinates are learned; individual dimensions rarely have a predefined human
meaning. Geometry still gives useful relationships.

$$\|x\|_2=\sqrt{\sum_i x_i^2},\qquad x\cdot y=\sum_i x_i y_i$$

The dot product combines magnitude and alignment. Cosine similarity divides out the
magnitudes and lies in $[-1,1]$ for nonzero real vectors.


In [ ]:
import numpy as np
import torch

def dot(x: np.ndarray, y: np.ndarray) -> float:
    assert x.shape == y.shape and x.ndim == 1
    return float(sum(float(a * b) for a, b in zip(x, y)))

def l2_norm(x: np.ndarray) -> float:
    return float(np.sqrt(dot(x, x)))

def cosine(x: np.ndarray, y: np.ndarray, eps: float = 1e-12) -> float:
    return dot(x, y) / max(l2_norm(x) * l2_norm(y), eps)

x = np.array([3.0, 4.0])
y = np.array([4.0, -3.0])
print("norm:", l2_norm(x), "dot:", dot(x, y), "cosine:", cosine(x, y))
assert l2_norm(x) == 5.0 and dot(x, y) == 0.0


## 2. Projection decomposes a vector

Projection of $x$ onto nonzero $u$:

$$\operatorname{proj}_u(x)=\frac{x\cdot u}{u\cdot u}u$$

Then $x=\operatorname{proj}_u(x)+r$, and residual $r$ is orthogonal to $u$. This pattern
reappears in least squares, basis changes, and analysis of learned representations.


In [ ]:
def project(x: np.ndarray, onto: np.ndarray) -> np.ndarray:
    denominator = dot(onto, onto)
    if denominator == 0:
        raise ValueError("cannot project onto the zero vector")
    return (dot(x, onto) / denominator) * onto

x = np.array([3.0, 4.0])
axis = np.array([1.0, 1.0])
parallel = project(x, axis)
residual = x - parallel
print("parallel:", parallel, "residual:", residual)
np.testing.assert_allclose(dot(residual, axis), 0.0, atol=1e-12)


## 3. Batch geometry

For embeddings `E=[N,C]`, all pairwise dot products are `E @ E.T = [N,N]`. Normalize
each row first to obtain a cosine-similarity matrix. The diagonal should be one unless
a row is zero.


In [ ]:
torch.manual_seed(42)
embeddings = torch.randn(5, 8)  # [tokens,C]
unit = embeddings / embeddings.norm(dim=-1, keepdim=True).clamp_min(1e-12)
similarities = unit @ unit.T    # [tokens,tokens]

print(similarities.round(decimals=2))
torch.testing.assert_close(similarities.diag(), torch.ones(5))
torch.testing.assert_close(similarities, similarities.T)


## 4. Distance, similarity, and scaling

Euclidean distance is sensitive to scale; cosine similarity is scale-invariant. Neither
is universally correct. Attention uses scaled dot products because magnitudes carry
information but growing dimension otherwise increases logit variance.


In [ ]:
anchor = np.array([1.0, 1.0])
candidates = {
    "same direction, far": np.array([10.0, 10.0]),
    "nearby, different angle": np.array([1.0, 0.5]),
    "opposite": np.array([-1.0, -1.0]),
}
for name, candidate in candidates.items():
    distance = np.linalg.norm(anchor - candidate)
    print(f"{name:24} distance={distance:6.2f} cosine={cosine(anchor, candidate):6.2f}")


## Failure modes to recognize

- **Zero-vector normalization:** division produces NaN without an epsilon or explicit policy.
- **Cosine as distance:** direction-only similarity ignores useful magnitude information.
- **Wrong reduction axis:** similarities aggregate tokens instead of features.
- **Embedding literalism:** one learned coordinate is treated as a stable human concept.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Implement L1 and infinity norms and compare their unit-distance rankings.
2. Use Gram–Schmidt to turn two independent vectors into an orthonormal pair.
3. Find the closest pair among ten random embeddings using cosine similarity without loops over pairs.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can derive and verify norm, cosine similarity, orthogonality, and projection for individual and batched vectors.

**Next:** 12 — Matrices and linear transformations.
